In [13]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from ucimlrepo import fetch_ucirepo
from pathlib import Path

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [14]:
# Config paths
ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "data"
OUTPUT_DIR = ROOT_DIR / "output"

## Part1: Feed-Forward Neural Network

Instructions:
- Build feed-forward neural network from scrach using numpy
- Cannot use scikit-learn, torch, keras for NN
- Scikit-learn can be used for data-processing, model evaluation


### 1.1 Dataset: Iris Dataset from UCI - ML Library

---
- Dataset [Link](https://archive.ics.uci.edu/dataset/53/iris)
- Features: sepal length, sepal width, petal length, petal width
- Class: Iris Setosa, Iris Versicolour, or Iris Virginica

---

In [15]:
# fetch dataset
iris = fetch_ucirepo(id=53)

# data (as pandas dataframes)
X = iris.data.features  # type: ignore
y = iris.data.targets  # type: ignore

# metadata
display(iris.metadata)

# variable information
display(iris.variables)


{'uci_id': 53,
 'name': 'Iris',
 'repository_url': 'https://archive.ics.uci.edu/dataset/53/iris',
 'data_url': 'https://archive.ics.uci.edu/static/public/53/data.csv',
 'abstract': 'A small classic dataset from Fisher, 1936. One of the earliest known datasets used for evaluating classification methods.\n',
 'area': 'Biology',
 'tasks': ['Classification'],
 'characteristics': ['Tabular'],
 'num_instances': 150,
 'num_features': 4,
 'feature_types': ['Real'],
 'demographics': [],
 'target_col': ['class'],
 'index_col': None,
 'has_missing_values': 'no',
 'missing_values_symbol': None,
 'year_of_dataset_creation': 1936,
 'last_updated': 'Tue Sep 12 2023',
 'dataset_doi': '10.24432/C56C76',
 'creators': ['R. A. Fisher'],
 'intro_paper': {'ID': 191,
  'type': 'NATIVE',
  'title': 'The Iris data set: In search of the source of virginica',
  'authors': 'A. Unwin, K. Kleinman',
  'venue': 'Significance, 2021',
  'year': 2021,
  'journal': 'Significance, 2021',
  'DOI': '1740-9713.01589',
  'UR

,name,role,type,demographic,description,units,missing_values
0,sepal length,Feature,Continuous,None,NaN,cm,no
1,sepal width,Feature,Continuous,None,NaN,cm,no
2,petal length,Feature,Continuous,None,NaN,cm,no
3,petal width,Feature,Continuous,None,NaN,cm,no
4,class,Target,Categorical,None,"class of iris plant: Iris Setosa, Iris Versico...",NaN,no


In [16]:
# Number of Instances
assert len(X) == len(y), "Feature / Class must have the same number of data."
print(f"Number of data:= {len(X)}")


Number of data:= 150


In [17]:
# Combine X and y to easily filter rows
df = pd.concat([X, y], axis=1)

# Drop 'Iris-virginica' to create a binary dataset
df_binary = df[df['class'] != 'Iris-virginica']

# Separate the features and target again
X_binary = df_binary.drop('class', axis=1)
y_binary = df_binary['class']

print(f"Data instances after filtering: {len(X_binary)}")
print(f"Classes remaining: {y_binary.unique()}")

# display(df_binary, X_binary, y_binary)

Data instances after filtering: 100
Classes remaining: <StringArray>
['Iris-setosa', 'Iris-versicolor']
Length: 2, dtype: str


###  1.2 Data Exploration

**Interpretation of the Labels:**
In this binary classification task, our labels represent the specific species of an Iris flower. After filtering out the 'Iris-virginica' class to balance the dataset, our model's goal is to predict whether a given flower is an **Iris-setosa** or an **Iris-versicolor** based on its physical characteristics.

**Included Features:**
The dataset includes four continuous numerical features, all measured in centimeters:
* **Sepal Length:** The length of the flower's sepal.
* **Sepal Width:** The width of the flower's sepal.
* **Petal Length:** The length of the flower's petal.
* **Petal Width:** The width of the flower's petal.

By feeding these four measurements into our models, we are training them to find the mathematical boundary that separates the Setosa species from the Versicolor species.

In [18]:
# Create the 80/20 train and test splits
X_train, X_test, y_train, y_test = train_test_split(X_binary, y_binary, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")


Training set size: 80
Testing set size: 20


In [19]:
# Initialize and train the baseline model
logreg_model = LogisticRegression()
logreg_model.fit(X_train, y_train)

# Predict on the test set and calculate accuracy
y_pred = logreg_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, y_pred)
# Convert 'Iris-setosa' -> 0, 'Iris-versicolor' -> 1

print(f"Logistic Regression Baseline Accuracy: {baseline_accuracy * 100:.2f}%")


Logistic Regression Baseline Accuracy: 100.00%


### 1.3 Data Preparation

Convert the string labels to binary integers (0 and 1) and reshape them into column vectors for matrix multiplication.

Convert pandas DataFrames to pure numpy arrays to prepare for math from scratch.


In [20]:
y_train_num = np.where(y_train == 'Iris-setosa', 0, 1)
y_test_num = np.where(y_test == 'Iris-setosa', 0, 1)

# Reshape to column vectors (n_samples x 1)
y_train_num = y_train_num.reshape(-1, 1)
y_test_num = y_test_num.reshape(-1, 1)

# Convert DataFrames to pure numpy arrays
X_train_np = X_train.to_numpy()
X_test_np = X_test.to_numpy()

print(f"X_train shape: {X_train_np.shape}, y_train shape: {y_train_num.shape}")
print(f"X_test shape: {X_test_np.shape}, y_test shape: {y_test_num.shape}")


X_train shape: (80, 4), y_train shape: (80, 1)
X_test shape: (20, 4), y_test shape: (20, 1)


### 1.4 Define Neural Network Math

Create the `sigmoid` activation function to squash outputs between 0 and 1.

Create the `train_neural_network` logic to initialize weights, execute forward/backward propagation, and apply gradient descent.


In [21]:
def sigmoid(z: np.ndarray | float) -> np.ndarray:
    """Compute the sigmoid activation for each input value.

    Args:
        z (np.ndarray | float): Scalar or array of logits to squash between 0 and 1.

    Returns:
        np.ndarray: Sigmoid values with the same shape as `z`.
    """
    return 1 / (1 + np.exp(-z))

In [22]:
def train_neural_network(
        X: np.ndarray,
        y: np.ndarray,
        learning_rate: float = 0.01,
        epochs: int = 1000,
) -> tuple[np.ndarray, float]:
    """Train a single-layer neural network from scratch.

    Args:
        X (np.ndarray): Feature matrix of shape (n_samples, n_features).
        y (np.ndarray): Column vector of binary targets with shape (n_samples, 1).
        learning_rate (float): Step size used when updating the parameters.
        epochs (int): Number of gradient descent iterations.

    Returns:
        tuple[np.ndarray, float]: Tuple containing the learned weights matrix and bias scalar.
    """
    np.random.seed(42)  # For reproducibility

    n_samples, n_features = X.shape
    weights = np.random.randn(n_features, 1) * 0.01  # Small random start for symmetry breaking
    bias = 0.0

    for _ in range(epochs):
        # Forward Pass
        Z = np.dot(X, weights) + bias
        A = sigmoid(Z)

        # Backward Pass (Error calculation)
        dZ = A - y

        # Calculate Gradients
        dW = (1 / n_samples) * np.dot(X.T, dZ)
        db = (1 / n_samples) * np.sum(dZ)

        # Update Weights and Bias via gradient descent
        weights -= learning_rate * dW
        bias -= learning_rate * db

    return weights, bias

### 1.5 Execute Training

Run the `train_neural_network` function using our prepared training numpy arrays.

We set epochs to 2,000 and the learning rate to 0.1 to ensure the model converges.



In [23]:
print("Starting training...")

trained_weights, trained_bias = train_neural_network(
    X_train_np,
    y_train_num,
    learning_rate=0.1,
    epochs=2000
)

print("Training complete!\n")
print(f"Optimized Weights:\n{trained_weights}")
print(f"Optimized Bias: {trained_bias}")


Starting training...
Training complete!

Optimized Weights:
[[-0.63153916]
 [-2.31140991]
 [ 3.56695212]
 [ 1.61289597]]
Optimized Bias: -0.4233844490087262


### 1.6  Evaluate and Compare Models

Create a `predict` function to classify the unseen test data using our trained weights.

Calculate the deep learning accuracy and print the comparison to the Logistic Regression baseline.

In [24]:
def predict(
        X: np.ndarray,
        weights: np.ndarray,
        bias: float,
) -> np.ndarray:
    """Predict binary classes using the trained parameters.

    Args:
        X (np.ndarray): Feature matrix to classify.
        weights (np.ndarray): Learned weights from training.
        bias (float): Learned bias term from training.

    Returns:
        np.ndarray: Binary predictions with the same number of rows as `X`.
    """
    Z = np.dot(X, weights) + bias
    A = sigmoid(Z)
    return (A >= 0.5).astype(int)


# Predict and Evaluate
y_pred_nn = predict(X_test_np, trained_weights, trained_bias)
nn_accuracy = accuracy_score(y_test_num, y_pred_nn)

print(f"Neural Network Accuracy: {nn_accuracy * 100:.2f}%")

try:
    print(f"Baseline Logistic Regression Accuracy: {baseline_accuracy * 100:.2f}%")
except NameError:
    print("Run the previous logistic regression block first to see comparison.")
# Task Configuration


Neural Network Accuracy: 100.00%
Baseline Logistic Regression Accuracy: 100.00%


### 1.7 Explaining "Self-Learning" to Management

**Memo: Understanding our "Self-Learning" Neural Network**

Since you are already familiar with linear regression, the concept of a neural network is actually not too far off!

In linear regression, we try to draw a "line of best fit" through our data. We measure how far off our line is from the actual data points (the error), and we adjust the slope and intercept to make that error as small as possible.

A neural network does exactly this, but on a much larger, more complex scale. Instead of just one slope and one intercept, it has hundreds or thousands of internal "dials" (which we call weights).

Here is what we mean when we say the AI is "self-learning" through a process called **backpropagation**:
1. **The Guess (Forward Pass):** The network looks at a piece of data and makes a prediction.
2. **The Reality Check (Loss Calculation):** It checks that prediction against the actual correct answer and calculates exactly how wrong it was.
3. **The Learning (Backpropagation):** This is the "self-learning" part. The algorithm works *backwards* through its internal dials. It calculates mathematically which dials need to be turned up, and which need to be turned down, to make a better guess next time.

It repeats this guessing and adjusting loop thousands of times until it has fine-tuned its dials to make highly accurate predictions. So, "self-learning" just means the system is automatically learning from its own mistakes and updating its internal math without us having to manually program the rules!

### ========= End of Part 1 ========= ###

## Part2: Transformers

### Encoder from Scratch for Sequence Reversal

In this part of notebook, We implement a basic transformer encoder from scratch in PyTorch to understand how transformers work under the hood. The task is to train the model to reverse a sequence of five digits, where each digit is an integer from 1 to 9.

For example:

Input: `[3, 5, 2, 4, 1]`
Output: `[1, 4, 2, 5, 3]`

The transformer is implemented using only basic PyTorch primitives such as `nn.Linear`, `nn.LayerNorm`, `nn.Parameter`, tensor operations, optimization tools, and loss functions. High-level transformer components such as `nn.Embedding`, `nn.MultiheadAttention`, and `nn.TransformerEncoder` are not used.

The notebook is organized step by step, including:
- implementation of each transformer component,
- generation of synthetic data,
- training,
- prediction,
- evaluation,
- and an architecture diagram.

### 1 Define the configuration for the Sequence-Reversal Task

The task is to train a transformer encoder to reverse sequences of length 5. Each token in the sequence is a digit from 1 to 9. For example, the input sequence `[3, 5, 2, 4, 1]` should produce the output `[1, 4, 2, 5, 3]`.

Since the digits already behave like discrete tokens, we can represent each sequence directly as a list of integers. The target sequence is simply the reversed version of the input.

For this implementation:
- sequence length = 5
- valid tokens = 1 to 9
- vocabulary size = 10, reserving index 0 as an unused padding-style slot
- output classes = 10, so the model predicts one token at each output position

In [25]:
SEQUENCE_LENGTH: int = 5
MIN_DIGIT: int = 1
MAX_DIGIT: int = 9
VOCAB_SIZE: int = 10  # digits 0-9, where 0 is unused
NUM_CLASSES: int = VOCAB_SIZE
D_MODEL: int = 32


In [26]:
print(f"Sequence length: {SEQUENCE_LENGTH}")
print(f"Valid digits: {MIN_DIGIT} - {MAX_DIGIT}")
print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Number of output classes: {NUM_CLASSES}")


Sequence length: 5
Valid digits: 1 - 9
Vocabulary size: 10
Number of output classes: 10


## 2. Generate the Synthetic Dataset

We generate a synthetic dataset for the sequence-reversal task. Each input sequence contains 5 digits, where each digit is an integer from 1 to 9. The target sequence is simply the reversed version of the input.

For example:

- Input: `[3, 5, 2, 4, 1]`
- Target: `[1, 4, 2, 5, 3]`

This dataset is simple enough to let us focus on implementing the transformer architecture from scratch, while still requiring the model to learn relationships between token positions.

Since the input data consists of integers between 1 and 9, it is already in a tokenized form. Therefore, no additional tokenization step is required. Each digit directly represents a token, where the integer value serves as the token index used by the embedding layer.

In [27]:
def generate_sequence(sequence_length: int = SEQUENCE_LENGTH, min_digit: int = MIN_DIGIT, max_digit: int = MAX_DIGIT) -> \
        tuple[list[int], list[int]]:
    """Generate a random digit sequence and its reversed target sequence."""
    sequence: list[int] = [random.randint(min_digit, max_digit) for _ in range(sequence_length)]
    reversed_sequence: list[int] = sequence[::-1]
    return sequence, reversed_sequence


def generate_dataset(num_samples: int, sequence_length: int = SEQUENCE_LENGTH, min_digit: int = MIN_DIGIT,
                     max_digit: int = MAX_DIGIT) -> tuple[torch.Tensor, torch.Tensor]:
    """Generate input and target tensors for the sequence-reversal task."""
    inputs: list[list[int]] = []
    targets: list[list[int]] = []

    for _ in range(num_samples):
        x, y = generate_sequence(sequence_length, min_digit, max_digit)
        inputs.append(x)
        targets.append(y)

    return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)


# Generate training and test sets
X_train, y_train = generate_dataset(5000)
X_test, y_test = generate_dataset(1000)

print(f"Training input shape : {X_train.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Test input shape     : {X_test.shape}")
print(f"Test target shape    : {y_test.shape}")

# Show a few examples
for i in range(3):
    print(f"Example {i + 1}")
    print(f"Input : {X_train[i].tolist()}")
    print(f"Target: {y_train[i].tolist()}")
    print()


Training input shape : torch.Size([5000, 5])
Training target shape: torch.Size([5000, 5])
Test input shape     : torch.Size([1000, 5])
Test target shape    : torch.Size([1000, 5])
Example 1
Input : [2, 1, 5, 4, 4]
Target: [4, 4, 5, 1, 2]

Example 2
Input : [3, 2, 9, 2, 7]
Target: [7, 2, 9, 2, 3]

Example 3
Input : [1, 1, 2, 4, 4]
Target: [4, 4, 2, 1, 1]



## 3. Create DataLoaders for Training and Testing

We wrap the input and target tensors into PyTorch datasets and then create data loaders for batching. This allows us to train the model efficiently using mini-batches rather than processing the entire dataset at once.

We use shuffling for the training data so that the model sees the examples in a different order each epoch. For the test data, shuffling is not necessary.

In [28]:
# Create PyTorch datasets
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

# Create data loaders
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Inspect one batch
sample_x, sample_y = next(iter(train_loader))
print(f"Batch input shape : {sample_x.shape}")
print(f"Batch target shape: {sample_y.shape}")
print()
print("Sample batch input:")
print(f"{sample_x[:2]}")
print()
print("Sample batch target:")
print(f"{sample_y[:2]}")


Batch input shape : torch.Size([64, 5])
Batch target shape: torch.Size([64, 5])

Sample batch input:
tensor([[9, 1, 9, 2, 8],
        [1, 8, 3, 2, 6]])

Sample batch target:
tensor([[8, 2, 9, 1, 9],
        [6, 2, 3, 8, 1]])


### 2.4 Implement the Token Embedding Layer

The first step in the transformer is to convert each discrete token into a dense vector representation. Instead of using `nn.Embedding`, which is not allowed in this assignment, we implement the embedding manually using a learned weight matrix.

If the vocabulary size is $|\mathcal{V}|$ and the embedding dimension is $d_{\text{model}}$, then the embedding matrix has shape $(|\mathcal{V}|, d_{\text{model}})$. Each token index is used to look up one row of this matrix.

Importantly, this embedding matrix is **learnable**. It is initialized randomly and updated during training through backpropagation. As training progresses, the model learns meaningful vector representations for each token, allowing it to capture relationships between different digits in a continuous space.

The output shape is:

- Input: `(batch_size, sequence_length)`
- Output: `(batch_size, sequence_length, d_model)`

In [29]:
class TokenEmbedding(nn.Module):
    """Learn a token embedding vector for each digit in the vocabulary."""

    def __init__(self, vocab_size: int, d_model: int) -> None:
        super().__init__()
        self.embedding_weight = nn.Parameter(torch.randn(vocab_size, d_model) * 0.02) # why 0.02

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Look up embedding vectors for token indices."""
        return self.embedding_weight[x]


In [30]:
# Test the token embedding layer
token_embedding = TokenEmbedding(VOCAB_SIZE, D_MODEL)

test_input: torch.Tensor = torch.tensor([
    [3, 5, 2, 4, 1],
    [9, 1, 7, 3, 6]
], dtype=torch.long)

test_output: torch.Tensor = token_embedding(test_input)

print(f"Input shape : {test_input.shape}")
print(f"Output shape: {test_output.shape}")


Input shape : torch.Size([2, 5])
Output shape: torch.Size([2, 5, 32])


### 2.5 Implement the Positional Embedding Layer

Because a transformer does not process tokens sequentially like an RNN, it needs positional information to understand the order of tokens in the sequence. We therefore learn a positional embedding matrix and add it to the token embeddings.

If the maximum sequence length is $L$ and the embedding dimension is $d_{\text{model}}$, then the positional embedding matrix has shape $(L, d_{\text{model}})$. Each position in the sequence uses the corresponding row from this matrix.

Importantly, the positional embedding matrix is also **learnable**. It is initialized randomly and updated during training. This allows the model to learn how different positions in the sequence should be represented and how position influences the relationships between tokens.

The output shape is:

- Input: `(batch_size, sequence_length, d_model)`
- Output: `(batch_size, sequence_length, d_model)`

In [31]:
class PositionalEmbedding(nn.Module):
    """Add a learnable positional embedding to each token representation."""

    def __init__(self, max_sequence_length: int, d_model: int) -> None:
        super().__init__()
        self.position_weight = nn.Parameter(torch.randn(max_sequence_length, d_model) * 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Broadcast position embeddings across the batch and add them to x."""
        batch_size, sequence_length, d_model = x.shape

        positions = torch.arange(sequence_length, device=x.device)
        pos_embed = self.position_weight[positions]  # (SEQUENCE_LENGTH, d_model)
        pos_embed = pos_embed.unsqueeze(0).expand(batch_size, sequence_length, d_model)

        return x + pos_embed


In [32]:
# Test the positional embedding layer
positional_embedding = PositionalEmbedding(SEQUENCE_LENGTH, D_MODEL)

token_embedded: torch.Tensor = token_embedding(test_input)
position_embedded: torch.Tensor = positional_embedding(token_embedded)

print(f"Token embedding shape     : {token_embedded.shape}")
print(f"Position embedding shape  : {position_embedded.shape}")


Token embedding shape     : torch.Size([2, 5, 32])
Position embedding shape  : torch.Size([2, 5, 32])


## 6. Combine Token and Positional Embeddings

The input representation for the transformer is obtained by adding the token embedding and the positional embedding. The token embedding tells the model what the symbol is, while the positional embedding tells the model where that symbol appears in the sequence.

This combined representation is the input that will be passed into the self-attention layer.

The output shape is:

- Input: `(batch_size, sequence_length)`
- Output: `(batch_size, sequence_length, d_model)`

In [33]:
class InputEmbedding(nn.Module):
    """Combine token and positional embeddings into one input representation."""

    def __init__(self, vocab_size: int, max_sequence_length: int, d_model: int) -> None:
        super().__init__()
        self.token_embedding = TokenEmbedding(vocab_size, d_model)
        self.positional_embedding = PositionalEmbedding(max_sequence_length, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return token embeddings enriched with positional information."""
        x = self.token_embedding(x)  # (batch_size, sequence_length, d_model)
        x = self.positional_embedding(x)  # (batch_size, sequence_length, d_model)
        return x


In [34]:
# Test the full input embedding block
input_embedding = InputEmbedding(VOCAB_SIZE, SEQUENCE_LENGTH, D_MODEL)
embedded_output: torch.Tensor = input_embedding(test_input)

print(f"Input shape : {test_input.shape}")
print(f"Output shape: {embedded_output.shape}")


Input shape : torch.Size([2, 5])
Output shape: torch.Size([2, 5, 32])


## 7. Implement the Self-Attention Layer

Self-attention allows each token in the sequence to compare itself with every other token and decide which positions are most relevant. This enables the transformer to model relationships between tokens without using recurrence.

To compute self-attention, we transform the input into three matrices:

- Query ($Q$)
- Key ($K$)
- Value ($V$)

These are learned linear transformations of the input.

We compute attention scores as:

$\text{AttentionScores} = \frac{QK^T}{\sqrt{d_k}}$

Then we apply the softmax function to obtain attention weights:

$\text{AttentionWeights} = \text{softmax}(\text{AttentionScores})$

Finally, we compute the output as a weighted sum of the values:

$\text{Output} = \text{AttentionWeights} \cdot V$

In this implementation, we also return the attention weights to help us inspect how the model distributes attention across the sequence.

The output shapes are:

- Output: `(batch_size, sequence_length, d_model)`
- Attention weights: `(batch_size, sequence_length, sequence_length)`

In [35]:
class SelfAttention(nn.Module):
    """Compute single-head self-attention over a sequence."""

    def __init__(self, d_model: int) -> None:
        super().__init__()
        self.d_model = d_model

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Return the attention output and attention weights."""
        Q = self.W_q(x)  # (batch_size, SEQUENCE_LENGTH, d_model)
        K = self.W_k(x)  # (batch_size, SEQUENCE_LENGTH, d_model)
        V = self.W_v(x)  # (batch_size, SEQUENCE_LENGTH, d_model)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_model)  # why -2, -1
        attention_weights = torch.softmax(scores, dim=-1)  # why -1
        output = torch.matmul(attention_weights, V)

        return output, attention_weights

In [36]:
# Test the updated self-attention layer
self_attention = SelfAttention(D_MODEL)

attention_output, attention_weights = self_attention(embedded_output)

print(f"Attention output shape : {attention_output.shape}")
print(f"Attention weights shape: {attention_weights.shape}")
print()
print(f"Attention weights for the first example:\n{attention_weights[0]}")


Attention output shape : torch.Size([2, 5, 32])
Attention weights shape: torch.Size([2, 5, 5])

Attention weights for the first example:
tensor([[0.2002, 0.2000, 0.1997, 0.2001, 0.1999],
        [0.2002, 0.2001, 0.1997, 0.2001, 0.1999],
        [0.2003, 0.2001, 0.1997, 0.2002, 0.1997],
        [0.2003, 0.2001, 0.1997, 0.2001, 0.1998],
        [0.2002, 0.2000, 0.1997, 0.2002, 0.1998]], grad_fn=<SelectBackward0>)


## 8. Implement the Feed-Forward Network

After the self-attention layer, each token representation is passed through a feed-forward neural network (FFN). This network is applied independently to each position in the sequence.

The feed-forward network consists of two linear layers with a non-linear activation function in between. In this implementation, we use the ReLU activation function.

The transformation is:

$\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$

This allows the model to further process and transform the information gathered by the attention mechanism.

The output shape is:

- Input: `(batch_size, sequence_length, d_model)`
- Output: `(batch_size, sequence_length, d_model)`

In [37]:
class FeedForward(nn.Module):
    """Apply a position-wise feed-forward transformation."""

    def __init__(self, d_model: int, d_ff: int = 64) -> None:
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Project tokens to a hidden layer, apply ReLU, and project back."""
        x = self.linear1(x)
        x = F.relu(x)
        x = self.linear2(x)
        return x

In [38]:
# Test the feed-forward network
ffn = FeedForward(D_MODEL)

ffn_output: torch.Tensor = ffn(attention_output)

print(f"Input shape : {attention_output.shape}")
print(f"Output shape: {ffn_output.shape}")


Input shape : torch.Size([2, 5, 32])
Output shape: torch.Size([2, 5, 32])


## 9. Add Residual Connections and Layer Normalization

A transformer encoder uses residual connections and layer normalization around both the self-attention layer and the feed-forward network. These operations help stabilize training and preserve useful information from earlier representations.

A residual connection adds the input of a sublayer to its output:

$\text{ResidualOutput} = x + \text{sublayer}(x)$

In this context, the **sublayer** refers to the transformation applied at that stage of the encoder. In our implementation, the two sublayers are:
- the self-attention layer
- the feed-forward network

After applying the residual connection, layer normalization is used to normalize the resulting vector. For each token representation $x \in \mathbb{R}^{d_{\text{model}}}$, layer normalization is computed as:

$\mu = \frac{1}{d_{\text{model}}} \sum_{i=1}^{d_{\text{model}}} x_i$

$\sigma^2 = \frac{1}{d_{\text{model}}} \sum_{i=1}^{d_{\text{model}}} (x_i - \mu)^2$

$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$

$y_i = \gamma_i \hat{x}_i + \beta_i$

where:
- $\mu$ is the mean of the features
- $\sigma^2$ is the variance
- $\gamma$ and $\beta$ are learnable parameters that scale and shift the normalized values

The final output of this block is:

$\text{LayerNorm}(x + \text{sublayer}(x))$

This operation ensures that the representation remains stable while allowing the model to refine the input through each sublayer.

In [39]:
class AddNorm(nn.Module):
    """Apply a residual connection followed by layer normalization."""

    def __init__(self, d_model: int) -> None:
        super().__init__()
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor, sublayer_output: torch.Tensor) -> torch.Tensor:
        """Add the residual branch to x and normalize the result."""
        return self.layer_norm(x + sublayer_output)

In [40]:
# Test AddNorm
add_norm = AddNorm(D_MODEL)

attention_input: torch.Tensor = embedded_output
normalized_output: torch.Tensor = add_norm(attention_input, attention_output)

print(f"Input shape          : {attention_input.shape}")
print(f"Sublayer output shape: {attention_output.shape}")
print(f"Normalized shape     : {normalized_output.shape}")


Input shape          : torch.Size([2, 5, 32])
Sublayer output shape: torch.Size([2, 5, 32])
Normalized shape     : torch.Size([2, 5, 32])


## 10. Implement the Encoder Block

We now combine the main transformer encoder components into a single encoder block. This block includes:

- self-attention
- residual connection followed by layer normalization
- feed-forward network
- another residual connection followed by layer normalization

The encoder block processes the sequence in two main stages. First, self-attention allows each token to gather information from all other tokens. Then, the feed-forward network further transforms each token representation independently.

The input and output of the encoder block both have shape:

- `(batch_size, sequence_length, d_model)`

In this implementation, we also return the attention weights so that we can inspect them later.

In [41]:
class EncoderBlock(nn.Module):
    """Run self-attention and feed-forward sublayers with residual connections."""

    def __init__(self, d_model: int, d_ff: int = 64) -> None:
        super().__init__()
        self.self_attention = SelfAttention(d_model)
        self.add_norm1 = AddNorm(d_model)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.add_norm2 = AddNorm(d_model)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Return encoded tokens and the attention weights from self-attention."""
        attention_output, attention_weights = self.self_attention(x)
        x = self.add_norm1(x, attention_output)

        ffn_output = self.feed_forward(x)
        x = self.add_norm2(x, ffn_output)

        return x, attention_weights


In [42]:
# Test the encoder block
encoder_block = EncoderBlock(D_MODEL, d_ff=64)

encoder_output: torch.Tensor
encoder_attention_weights: torch.Tensor
encoder_output, encoder_attention_weights = encoder_block(embedded_output)

print(f"Encoder output shape   : {encoder_output.shape}")
print(f"Attention weights shape: {encoder_attention_weights.shape}")

Encoder output shape   : torch.Size([2, 5, 32])
Attention weights shape: torch.Size([2, 5, 5])


## 11. Build the Full Transformer Encoder Model

We now combine all the previously defined components into the complete transformer model for the sequence-reversal task.

The model consists of:

- an input embedding layer, which combines token and positional embeddings
- one encoder block
- a final linear layer (perceptron) that maps each token representation to output scores over the vocabulary

For each position in the sequence, the model outputs a vector of logits of size equal to the vocabulary size. These logits are then used during training with a cross-entropy loss function.

The output shape of the model is:

- Input: `(batch_size, sequence_length)`
- Output logits: `(batch_size,sequence_length, vocab_size)`

In [43]:
class TransformerEncoderModel(nn.Module):
    """Map input digit sequences to per-position vocabulary logits."""

    def __init__(self, vocab_size: int, max_sequence_length: int, d_model: int, d_ff: int = 64) -> None:
        super().__init__()
        self.input_embedding = InputEmbedding(vocab_size, max_sequence_length, d_model)
        self.encoder_block = EncoderBlock(d_model, d_ff)
        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Return logits for each position and the encoder attention weights."""
        x = self.input_embedding(x)                     # (batch_size, sequence_length, d_model)
        x, attention_weights = self.encoder_block(x)    # (batch_size, sequence_length, d_model)
        logits = self.output_layer(x)                   # (batch_size, sequence_length, vocab_size)
        return logits, attention_weights

In [44]:
# Test the full model
model = TransformerEncoderModel(
    vocab_size=VOCAB_SIZE,
    max_sequence_length=SEQUENCE_LENGTH,
    d_model=D_MODEL,
    d_ff=64
).to(device)

test_batch: torch.Tensor = sample_x.to(device)
logits: torch.Tensor
attention_weights: torch.Tensor
logits, attention_weights = model(test_batch)

print(f"Input shape            : {test_batch.shape}")
print(f"Logits shape           : {logits.shape}")
print(f"Attention weights shape: {attention_weights.shape}")


Input shape            : torch.Size([64, 5])
Logits shape           : torch.Size([64, 5, 10])
Attention weights shape: torch.Size([64, 5, 5])


## 12. Define the Loss Function and Optimizer

To train the model, we use the cross-entropy loss function. Since the model predicts one token at each position in the sequence, we compare the predicted logits with the target class at each position.

The model output has shape `(batch_size, sequence_length, vocab_size)`, while the target has shape `(batch_size, sequence_length)`. For `CrossEntropyLoss`, we reshape the logits so that each token prediction is treated as one training example.

Although the model outputs logits (unnormalized scores), these are internally converted into probabilities using the softmax function. For a vector of logits $z$, softmax is defined as:

$p_i = \frac{e^{z_i}}{\sum_{j} e^{z_j}}$

This transforms the logits into a probability distribution over the vocabulary, where all values are positive and sum to 1.

During training, the model focuses on the probability assigned to the correct class. If the predicted probability for the correct token is high, the loss is low. If the probability is low, the loss increases significantly, penalizing the model. This encourages the model to assign higher probability to the correct class over time.

In PyTorch, `nn.CrossEntropyLoss()` combines the softmax operation and the negative log-likelihood loss in a numerically stable way, so we do not need to apply softmax explicitly in the code.

We also define the optimizer that will update the model parameters during training. In this implementation, we use the Adam optimizer.

In [45]:
criterion: nn.CrossEntropyLoss = nn.CrossEntropyLoss()
optimizer: torch.optim.Optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Loss function: {criterion}")
print(f"Optimizer: {optimizer}")


Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


## 13. Train the Transformer Model

We now train the transformer encoder on the synthetic sequence-reversal dataset.

For each mini-batch, we:

1. move the input and target tensors to the selected device
2. run a forward pass through the model
3. reshape the logits and targets for the loss function
4. compute the loss
5. backpropagate the gradients
6. update the model parameters

We record the average training loss for each epoch so that we can later visualize how the model learns over time.

In [ ]:
NUM_EPOCHS: int = 20
train_losses: list[float] = []

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss: float = 0.0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        logits, _ = model(batch_x)
        # logits shape: (batch_size, SEQUENCE_LENGTH, vocab_size)

        loss: torch.Tensor = criterion(
            logits.view(-1, VOCAB_SIZE),  # (batch_size * SEQUENCE_LENGTH, vocab_size)
            batch_y.view(-1)  # (batch_size * SEQUENCE_LENGTH)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss: float = total_loss / len(train_loader)
    train_losses.append(avg_loss)

    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}, Training Loss: {avg_loss:.4f}")


Epoch 1/20, Training Loss: 2.0397
Epoch 2/20, Training Loss: 1.4257
Epoch 3/20, Training Loss: 0.9068
Epoch 4/20, Training Loss: 0.1901
Epoch 5/20, Training Loss: 0.0112
Epoch 6/20, Training Loss: 0.0064
Epoch 7/20, Training Loss: 0.0045
Epoch 8/20, Training Loss: 0.0034
Epoch 9/20, Training Loss: 0.0026
Epoch 10/20, Training Loss: 0.0021
Epoch 11/20, Training Loss: 0.0018
Epoch 12/20, Training Loss: 0.0015
Epoch 13/20, Training Loss: 0.0013
Epoch 14/20, Training Loss: 0.0011
Epoch 15/20, Training Loss: 0.0010


## 14. Plot the Training Loss Curve

To monitor learning, we plot the average training loss at each epoch. A decreasing loss indicates that the model is learning to map input sequences to their reversed outputs.

This plot helps us verify whether training is progressing as expected.

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(range(1, NUM_EPOCHS + 1), train_losses, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss Over Epochs")
plt.grid(True)
plt.savefig(OUTPUT_DIR / "transformers_training_loss.png", dpi=300)
plt.show()


## 15. Generate Predictions with the Trained Model

We define a function to generate predictions from the trained model. Given an input sequence, the model outputs logits for each position, representing scores for each possible token in the vocabulary.

To obtain the final predicted token at each position, we apply the argmax function, which selects the index of the largest value in the logits vector. This corresponds to the token with the highest predicted probability.

Although logits are not probabilities, applying softmax would preserve the ordering of values, so the index of the maximum logit is the same as the index of the maximum probability.

This allows us to convert the model output into a sequence of predicted tokens, which we can compare to the expected reversed sequence.

In [ ]:
def predict(model: nn.Module, input_sequence: list[int]) -> list[int]:
    """Predict the output sequence for a single input sequence."""
    model.eval()

    x: torch.Tensor = torch.tensor([input_sequence], dtype=torch.long).to(device)

    with torch.no_grad():
        logits, _ = model(x)

    predictions: torch.Tensor = torch.argmax(logits, dim=-1)

    return predictions[0].cpu().tolist()


# Use real test examples and display nicely
num_examples: int = 5
rows: list[dict[str, object]] = []

for i in range(num_examples):
    input_seq = X_test[i].tolist()
    true_seq = y_test[i].tolist()
    pred_seq = predict(model, input_seq)

    rows.append({
        "Input": input_seq,
        "Predicted": pred_seq,
        "Expected": true_seq,
        "Correct": pred_seq == true_seq
    })

print(f"Prepared {num_examples} example rows for display.")
df = pd.DataFrame(rows)
df


## 16. Evaluate the Model Quantitatively

To evaluate the model, we measure how well it predicts reversed sequences on the test set.

We use two metrics:

- **token-level accuracy**: the proportion of individual positions predicted correctly
- **sequence-level accuracy**: the proportion of entire sequences predicted perfectly

Token-level accuracy tells us how often the model gets individual digits right, while sequence-level accuracy is stricter because all five positions must be correct for a sequence to count as correct.

In [ ]:
def evaluate_model(model: nn.Module, data_loader: DataLoader) -> tuple[float, float]:
    """Compute token-level and sequence-level accuracy for a data loader."""
    model.eval()

    total_tokens: int = 0
    correct_tokens: int = 0
    total_sequences: int = 0
    correct_sequences: int = 0

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            logits, _ = model(batch_x)
            predictions = torch.argmax(logits, dim=-1)

            # Token-level accuracy
            correct_tokens += (predictions == batch_y).sum().item()
            total_tokens += batch_y.numel()

            # Sequence-level accuracy
            sequence_matches = (predictions == batch_y).all(dim=1)
            correct_sequences += sequence_matches.sum().item()
            total_sequences += batch_y.size(0)

    token_accuracy: float = correct_tokens / total_tokens
    sequence_accuracy: float = correct_sequences / total_sequences

    return token_accuracy, sequence_accuracy


token_acc, sequence_acc = evaluate_model(model, test_loader)

print(f"Token-level accuracy   : {token_acc:.4f}")
print(f"Sequence-level accuracy: {sequence_acc:.4f}")


## 17. Commentary on the Results

The model achieved perfect performance on the test set, with both token-level accuracy and sequence-level accuracy equal to 1.0000.

This means that the transformer encoder successfully learned the sequence-reversal task for all test examples. At the token level, every individual digit was predicted correctly. At the sequence level, every entire sequence was reversed perfectly, indicating that the model learned the full transformation rather than partially approximating it.

These results demonstrate that the implemented architecture is capable of capturing positional relationships and dependencies between tokens effectively. The combination of token embeddings, positional embeddings, self-attention, and feed-forward layers allows the model to correctly identify how each position in the input maps to a position in the output.

Given that the task is relatively simple and the dataset is synthetic, achieving perfect accuracy is expected once the model is properly implemented and trained. This confirms that the transformer components were implemented correctly and that the training process was effective.

Overall, the results validate both the correctness of the implementation and the ability of transformer-based models to learn structured sequence transformations.

## 18. Transformer Architecture Diagram

```text

┌────────────────────────────┐   ┌────────────────────────────┐   ┌────────────────────────────┐   ┌────────────────────────────┐
│ 1. Input                   │ → │ 2. Token Embedding         │ → │ 3. Positional Embedding    │ → │ 4. Add Embeddings          │
│ (64, 5)                    │   │ (64, 5) → (64, 5, 32)      │   │ (64, 5, 32) → (64, 5, 32)  │   │ (64, 5, 32) → (64, 5, 32)  │
└────────────────────────────┘   └────────────────────────────┘   └────────────────────────────┘   └────────────────────────────┘
                                                                                                                 │
                                                                                                                 │
Encoder                                                                                                          ▼
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

┌────────────────────────────┐   ┌────────────────────────────┐   ┌────────────────────────────┐   ┌────────────────────────────┐
│ 8. Resid Conn & LayerNorm  │ ← │ 7. Feed-Forward Network    │ ← │ 6. Resid Conn & LayerNorm  │ ← │ 5. Self-Attention          │
│ (64, 5, 32) →              │   │ (64, 5, 32) →              │   │ (64, 5, 32) →              │   │ (64, 5, 32) →              │
│ (64, 5, 32)                │   │ (64, 5, 64) →              │   │ (64, 5, 32)                │   │ (64, 5, 32)                │
│                            │   │ (64, 5, 32)                │   │                            │   │                            │
└────────────────────────────┘   └────────────────────────────┘   └────────────────────────────┘   └────────────────────────────┘
          
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
          │     
          │
          ▼
┌────────────────────────────┐   ┌────────────────────────────┐
│ 9. Final Linear Perceptron │ → │ 10. Prediction (argmax)    │
│ (64, 5, 32) →              │   │ (64, 5, 10) → (64, 5)      │
│ (64, 5, 10)                │   │                            │
└────────────────────────────┘   └────────────────────────────┘


The tensor `(64, 5, 32)` can be interpreted as a 3D structure with 64 layers, where each layer is a matrix of 5 rows (tokens) and 32 columns (embedding dimensions).